# ESA Collision Avoidance Challenge
## Part 1: Problem Definition and Questions

## Project Roadmap

This notebook is **Part 1** of our four-part core engineering series (complemented by the full 6-Phase Research Archive in `research/`):

| Part | Notebook | Scope | Status |
| :--- | :--- | :--- | :--- |
| **1** | `01_problem_definition.ipynb` | Problem formulation, physical risk definition, causal rules, and official ESA metric | **Current** |
| **2** | `02_eda.ipynb` | Deep exploratory data analysis, statistical intervals, and orbital physics checks | Next |
| **3** | `03_preprocessing_fe.ipynb` | Causal event-level feature engineering ($T \ge 2.0\text{ days}$) and invariance testing | Upcoming |
| **4** | `04_modeling.ipynb` | Baseline benchmarking, gradient boosting, two-stage physics-guided architecture (`M6`/`M7`), and bundle export | Upcoming |

## Three-Layer Engineering Methodology

Every concept in this project is developed across three complementary layers:

| Layer | Core Question Answered |
| :--- | :--- |
| **1. Physical Intuition** | Why does this phenomenon occur in Low Earth Orbit (LEO), and what is the physical mental model? |
| **2. Mathematical Rigor** | What is the exact equation, statistical estimator, or optimization objective? |
| **3. Implementation & Verification** | How do we translate the math into vectorized Python/NumPy code and prove its correctness via unit/reference tests? |

We also highlight a fourth takeaway on every block — **Operational Insight**: the non-obvious finding that directly governs an architectural decision.

## 1. Operational Problem Statement

An active spacecraft orbits the Earth surrounded by tens of thousands of tracked space debris fragments and other satellites. Ground-based radar and optical networks (such as the US Space Surveillance Network) continuously track these objects. Whenever a secondary object ("chaser") is projected to pass close to a protected asset ("target"), a **Conjunction Data Message (CDM)** is issued:

- Each CDM records the encounter geometry at the **Time of Closest Approach (`TCA`)**, the state vectors and $6\times 6$ covariance matrices of both objects in the local Radial–Transverse–Normal (`RTN`) frame, and the computed collision probability (`risk`, expressed in $\log_{10} P_c$).
- For a single close approach (`event_id`), a time series of CDMs arrives over roughly 7 days (approximately 3 updates per day) as fresh tracking observations shrink the orbital uncertainty ellipsoid.
- The final CDM closest to `TCA` represents the best available knowledge of the conjunction (`final_risk`).
- **The 48-Hour Operational Cutoff Wall:** At the European Space Operations Centre (ESOC), flight dynamics engineers must begin planning a Collision Avoidance Manoeuvre (`CAM`) at least **2.0 days (48 hours)** before `TCA` (uplink scheduling, thruster alignment, and secondary conjunction screening).

> **Core Predictive Question:** Given only the sequence of CDMs available at least **2.0 days before TCA** ($\text{time\_to\_tca} \ge 2.0$), what will the final collision risk ($\text{final\_risk}$) be near `TCA`, and will the event escalate above the high-risk reaction threshold ($\text{risk} \ge -6.0$, i.e., $P_c \ge 10^{-6}$)?

## Mathematical Formulation of the Causal Prediction Task

- **Physical Intuition:** Each conjunction event $e$ is a time-ordered sequence of tracking updates. We stand at the $T - 2.0\text{ day}$ decision barrier and must forecast the terminal state of the series without ever peeking past the barrier.
- **Mathematical Definition:** Let event $e$ consist of $m_e$ chronologically ordered messages $(t_{e,1} > t_{e,2} > \dots > t_{e,m_e})$, where $t_{e,i}$ is `time_to_tca` in days, $\mathbf{x}_{e,i} \in \mathbb{R}^{103}$ is the CDM attribute vector, and $r_{e,i} = \log_{10} P_{c,i}$ is the reported risk:
  - **Target Label:** $y_e = r_{e,m_e}$ (the `risk` recorded in the final CDM closest to `TCA`, where $t_{e,m_e} < 1.0\text{ day}$ in test-eligible cohorts).
  - **Admissible Pre-Cutoff Information Set:** $\mathcal{I}_e = \{(\mathbf{x}_{e,i}, r_{e,i}) : t_{e,i} \ge 2.0\}$.
  - **Objective:** Learn a mapping $f(\mathcal{I}_e) \mapsto \hat{y}_e$ that minimizes the official asymmetric competition loss $\mathcal{L}(y, \hat{y}) = \frac{\text{MSE}_{\text{HR}}(y, \hat{y})}{F_2(y, \hat{y})}$.
- **Operational Insight:** Any feature or threshold derived from CDMs with $t_{e,i} < 2.0\text{ days}$ constitutes **future temporal leakage** and invalidates the model.

## What is `risk` Mathematically? (The Uncertainty Dilution Paradox)

- **Physical Intuition:** Neither the target spacecraft's position nor the chaser debris position is known with zero error at `TCA`. Around each object lies a 3D Gaussian probability cloud (the covariance ellipsoid). The collision probability $P_c$ is the integral of the combined relative position probability density over the combined Hard-Body Radius (`HBR`) sphere $R$.
- **Mathematical Formulation:** Projecting the 3D relative position error onto the 2D encounter plane (**B-plane**) perpendicular to the relative velocity vector $\mathbf{v}_{\text{rel}}$:
  $$\mathbf{r}_{\perp} \sim \mathcal{N}(\boldsymbol{\mu}_{\perp}, \boldsymbol{\Sigma}_{\perp}), \qquad P_c = \iint_{\|\mathbf{r}_{\perp}\| \le R} \frac{1}{2\pi |\boldsymbol{\Sigma}_{\perp}|^{1/2}} \exp\!\left(-\frac{1}{2}(\mathbf{r}_{\perp}-\boldsymbol{\mu}_{\perp})^\top \boldsymbol{\Sigma}_{\perp}^{-1}(\mathbf{r}_{\perp}-\boldsymbol{\mu}_{\perp})\right) d\mathbf{r}_{\perp}$$
  The dataset column `risk` is $\log_{10}(P_c)$, censored at a numerical floor of `-30.0`. Thus, the high-risk threshold $P_c \ge 10^{-6}$ corresponds to $\text{risk} \ge -6.0$.
- **The Dilution Region Paradox (Key Physics Insight):** Under an isotropic 2D approximation with miss distance $d = \|\boldsymbol{\mu}_{\perp}\|$, combined uncertainty $\sigma$, and small hard-body radius $R \ll \sigma$:
  $$P_c(\sigma) \approx \frac{R^2}{2\sigma^2} \exp\!\left(-\frac{d^2}{2\sigma^2}\right)$$
  Differentiating $\ln P_c(\sigma)$ with respect to $\sigma$ and setting the derivative to zero yields the worst-case covariance scale $\sigma^* = \frac{d}{\sqrt{2}}$, at which the maximum possible collision probability is achieved:
  $$P_{c,\max} = \frac{R^2}{e \cdot d^2} \qquad \Longleftrightarrow \qquad \text{max\_risk\_estimate} = \log_{10}(P_{c,\max})$$
  When early tracking uncertainty is very large ($\sigma \gg d$), $P_c$ is artificially small (**the dilution region**). As new radar tracks arrive and $\sigma$ shrinks toward $d/\sqrt{2}$, $P_c$ **climbs sharply** toward $P_{c,\max}$ before either dropping (if the miss distance holds) or confirming a true collision threat!

### Experiment 1: Analytical $P_c(\sigma)$ Curve vs. 2,000,000-Sample Monte Carlo Simulation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

d = 500.0                          # miss distance in meters
R = 20.0                           # combined hard-body radius in meters
sigmas = np.logspace(1, 4, 200)    # isotropic uncertainty from 10 m to 10 km

pc_formula = (R**2 / (2 * sigmas**2)) * np.exp(-d**2 / (2 * sigmas**2))

# Monte Carlo check at one sigma: sample the relative position and count hits inside the radius R
rng = np.random.default_rng(0)
sigma_check = 300.0
samples = rng.normal(loc=[d, 0.0], scale=sigma_check, size=(2_000_000, 2))
pc_mc = (np.linalg.norm(samples, axis=1) <= R).mean()
pc_f = (R**2 / (2 * sigma_check**2)) * np.exp(-d**2 / (2 * sigma_check**2))
print(f"sigma = {sigma_check:.0f} m | formula: {pc_f:.3e} | Monte Carlo: {pc_mc:.3e}")

sigma_best = d / np.sqrt(2)
print(f"peak at sigma = d / sqrt(2) = {sigma_best:.0f} m | Pc_max = R^2 / (e d^2) = {R**2 / (np.e * d**2):.3e}")

plt.figure(figsize=(7, 4))
plt.loglog(sigmas, pc_formula)
plt.axvline(sigma_best, color="gray", linestyle=":", label="sigma = d / sqrt(2)")
plt.axhline(1e-6, color="red", linestyle="--", label="threshold 1e-6")
plt.xlabel("uncertainty sigma (m)")
plt.ylabel("collision probability Pc")
plt.title(f"Pc versus uncertainty (miss distance = {d:.0f} m)")
plt.legend()
plt.show()

### Explanation of Experiment 1

- `np.logspace(1, 4, 200)` generates 200 logarithmically spaced uncertainty values $\sigma \in [10\text{ m}, 10,000\text{ m}]$.
- `pc_formula` evaluates the closed-form 2D Gaussian encounter integral across all 200 $\sigma$ values simultaneously.
- In the Monte Carlo verification step, `rng.normal(loc=[d, 0.0], scale=sigma_check, size=(2_000_000, 2))` draws 2,000,000 random conjunction realizations at $\sigma = 300\text{ m}$ and counts the fraction falling inside the hard-body radius $R = 10\text{ m}$.
- The exact agreement between `formula` and `Monte Carlo` confirms our physical equation.
- **How to Read the Plot:** Notice the bell-shaped curve in $\log_{10}\sigma$: an event on the far right (large $\sigma$, low `last_risk < -6.0`, but high peak `max_risk_estimate > -5.0`) sits in the **Dilution Region**. As new radar observations contract $\sigma$ from right to left, its risk will surge across the `-6.0` red threshold line! This exact physical mechanism motivates our **Stage-2 Risk-Blind Physical Rescue Classifier (`NORISK_NOCAT`)** and **`dilution_gap = max_risk_estimate - last_risk`** feature.

## 2. Official Competition Rules & Data Schema

| Component | Official Specification in Our Pipeline |
| :--- | :--- |
| **Prediction Target (`final_risk`)** | The `risk` ($\log_{10} P_c$) recorded in the final CDM closest to `TCA` for each event |
| **Admissible Model Inputs** | Only CDMs with `time_to_tca >= 2.0` days (at least 48 hours prior to `TCA`) |
| **Observational Unit** | The conjunction event (`event_id`), NOT individual CDM rows |
| **High-Risk Class Definition** | `final_risk >= -6.0` ($P_c \ge 10^{-6}$) |
| **Official Evaluation Metric** | $\mathcal{L} = \frac{\text{MSE}_{\text{HR}}}{F_2}$ with predictions clipped below at `-6.001` (lower $\mathcal{L}$ is better) |

**Why the Observational Unit Must Be the Event:** The raw training file contains `162,634` CDM rows across `13,154` unique events. Rows belonging to the same `event_id` share identical target/chaser satellites and highly correlated state trajectories; splitting rows randomly across train/validation folds would leak the exact same event into both sets.

## 3. Why the Asymmetric Composite Metric $\mathcal{L} = \text{MSE}_{\text{HR}} / F_2$?

The ESA Advanced Concepts Team designed a two-part composite loss function:
1. **High-Risk Mean Squared Error ($\text{MSE}_{\text{HR}}$):** Computed **exclusively** over events whose true final risk is high ($y_e \ge -6.0$). Errors on truly safe events ($y_e < -6.0$) do not enter $\text{MSE}_{\text{HR}}$, and all predicted values below `-6.001` are clipped to `-6.001` prior to scoring.
2. **Classification $F_2$-Score ($F_2$):** Evaluates binary alarm classification ($\hat{y}_e \ge -6.0$ vs. $y_e \ge -6.0$), weighting **Recall ($R$) four times as heavily as Precision ($P$)**:
   $$F_2 = \frac{5 \cdot P \cdot R}{4 \cdot P + R} = \frac{5 \cdot \text{TP}}{5 \cdot \text{TP} + 4 \cdot \text{FN} + \text{FP}}$$
3. **Composite Ratio ($\mathcal{L} = \text{MSE}_{\text{HR}} / F_2$):** Dividing by $F_2 \in (0, 1]$ penalizes missed high-risk conjunctions (`FN`) doubly — both by shrinking the denominator $F_2$ and by incurring a squared error $(y_e - (-6.001))^2$ in the numerator $\text{MSE}_{\text{HR}}$!

### Mathematical Properties of $\mathcal{L} = \text{MSE}_{\text{HR}} / F_2$

From the binary confusion matrix (`TP`, `FP`, `FN`, `TN`):
- $P = \frac{\text{TP}}{\text{TP} + \text{FP}}$, $R = \frac{\text{TP}}{\text{TP} + \text{FN}}$, and $\frac{1}{F_2} = \frac{0.2}{P} + \frac{0.8}{R}$.
- **Key Properties:**
  1. $\mathcal{L} \ge 0$ always (0 is a perfect oracle).
  2. If $\text{TP} = 0$, then $R = 0 \implies F_2 = 0 \implies \mathcal{L} = +\infty$.
  3. Under the official clipping rule ($\hat{y}_{\text{clipped}} = \max(\hat{y}, -6.001)$), any false negative (`FN`, where $y_e \ge -6.0$ but $\hat{y}_e < -6.0$) is evaluated at $-6.001$, contributing $(y_e + 6.001)^2$ to $\text{MSE}_{\text{HR}}$.
- **Worked Hand Calculation:** Suppose $y = [-8.0, -5.0, -4.0, -7.0]$ and $\hat{y} = [-5.5, -5.0, -3.0, -9.0]$ at threshold $-6.0$:
  - True High-Risk events: indices 1 and 2 ($y = -5.0, -4.0$). Predicted Alarms: indices 0, 1, and 2.
  - Thus $\text{TP} = 2, \text{FP} = 1, \text{FN} = 0 \implies P = 2/3, R = 1.0 \implies F_2 = \frac{5(2/3)(1)}{4(2/3)+1} = \frac{10}{11} \approx 0.9091$.
  - $\text{MSE}_{\text{HR}} = \frac{(-5.0 - (-5.0))^2 + (-4.0 - (-3.0))^2}{2} = 0.50 \implies \mathcal{L} = \frac{0.50}{10/11} = 0.55$.

## 4. Cohort Construction & Selection Bias Between `train_data` and `test_data`

According to the official ESA Kelvins dataset documentation:
1. Every event in `test_data.csv` (`2,167` events) was selected such that its final CDM reaches within 1 day of `TCA` ($\min(\text{time\_to\_tca}) < 1.0\text{ day}$), while only pre-cutoff CDMs ($\text{time\_to\_tca} \ge 2.0\text{ days}$) are provided in `test_data.csv`.
2. In contrast, `train_data.csv` (`13,154` raw events) includes both events that terminate early ($\min(\text{time\_to\_tca}) \ge 2.0\text{ days}$) and events that start late ($\max(\text{time\_to\_tca}) < 2.0\text{ days}$).
3. Therefore, to construct a **causally valid training and cross-validation cohort**, we filter `train_data.csv` to events that have at least one admissible input CDM at $\text{time\_to\_tca} \ge 2.0\text{ days}$ and a valid post-cutoff target (`8,892` causal training events, or `8,293` under the stricter $\min(\text{time\_to\_tca}) < 1.0\text{ day}$ filter).

## 5. Nine Guiding Research Questions (Answered Across Notebooks 01–04)

| ID | Research Question | Engineering Impact | Answered In |
| :--- | :--- | :--- | :--- |
| **Q1** | What is the exact positive class prevalence ($\text{final\_risk} \ge -6.0$) and censoring floor rate (`-30.0`)? | Governs class weighting, floor-indicator features, and threshold calibration | Part 1 & Part 2 |
| **Q2** | How many training events satisfy the causal cutoff and test-like horizon criteria? | Defines our `8,892` / `8,293` event validation cohorts | Part 1 |
| **Q3** | How strong is the `last_risk` persistence baseline (`B0`), and where does it fail? | Establishes the benchmark to beat and isolates Z2 False Alarms vs. Z3 False Negatives | Part 2 |
| **Q4** | How do collision risk and positional uncertainty ($\sigma_R, \sigma_T, \sigma_N$) evolve toward `TCA`? | Motivates log-uncertainty slope features (`slope_log_c_sigma_*`) | Part 2 & Part 3 |
| **Q5** | Which raw CDM attributes exhibit the strongest monotonic correlation with `final_risk`? | Identifies core predictors (`last_risk`, `max_risk_estimate`, `mahalanobis_distance`) | Part 2 |
| **Q6** | Are missing values structural (`MAR`/`MNAR`) or random (`MCAR`)? | Determines native tree missing-value routing vs. median imputation | Part 2 & Part 3 |
| **Q7** | Is there covariate/prior shift between `train_data.csv` and `test_data.csv`? | Reveals the $2.1546\times$ active-alarm density shift ($3.81\%$ vs. $8.21\%$) | Part 2 & Research Archive |
| **Q8** | Do `mission_id` and `c_object_type` carry generalizable signal or overfitting risk? | Motivates the `NOCAT` feature matrix that drops object-type one-hots | Part 2, Part 3 & Phase 3 |
| **Q9** | Which snapshot features are collinear twins ($|\rho_{\text{Spearman}}| > 0.95$)? | Guides collinearity pruning while protecting primary physical variables | Part 2, Part 3 & Part 4 |

## 6. Hands-On Data Audit & Metric Verification

Each code block below is paired with a technical walkthrough explaining the implementation details and verification checks.

### Block 1: Imports & Display Configuration

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import fbeta_score, mean_squared_error

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", "{:.6g}".format)

### Walkthrough of Block 1

- `numpy` (`np`) and `pandas` (`pd`) provide vectorized array and tabular operations across the 103 CDM columns.
- `fbeta_score` and `mean_squared_error` provide reference implementations to cross-check our custom vectorized ESA competition metric.
- `pd.set_option("display.max_columns", 120)` ensures all 103 CDM columns remain visible during schema inspection.

### Block 2: File Paths & Competition Constants

In [ ]:
from pathlib import Path

# look for the data folder next to the notebook or one level up
DATA_DIR = next((p for p in (Path("data"), Path("../data")) if p.exists()), Path("data"))
TRAIN_PATH = DATA_DIR / "train_data.zip"
TEST_PATH = DATA_DIR / "test_data.csv"

HIGH_RISK_THRESHOLD = -6.0         # risk is already log10, so 1e-6 equals -6
MIN_TIME_TO_TCA_FOR_INPUT = 2.0    # model inputs: CDMs at least 2 days before TCA
MAX_FINAL_TIME_TO_TCA = 1.0        # test events: last CDM within 1 day of TCA

### Walkthrough of Block 2

- `DATA_DIR` automatically resolves either `./data` or `../data` so the notebook runs seamlessly from both the repository root and the `notebooks/` folder.
- Centralizing `HIGH_RISK_THRESHOLD = -6.0`, `MIN_TIME_TO_TCA_FOR_INPUT = 2.0`, and `MAX_FINAL_TIME_TO_TCA = 1.0` guarantees zero discrepancy between exploratory notebooks, training scripts, and the Streamlit platform.

### Block 3: Loading the Raw CDM Datasets

In [ ]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print(f"train: {train.shape[0]:,} rows x {train.shape[1]} columns")
print(f"test : {test.shape[0]:,} rows x {test.shape[1]} columns")

### Walkthrough of Block 3

- `pd.read_csv(path)` transparently decompresses `.zip` archives or reads `.csv` files directly.
- Inspecting `train.shape` and `test.shape` immediately verifies that all `103` columns are loaded intact.

### Block 4: Verifying Dataset Dimensions Against Official ESA Documentation

In [ ]:
expected = {
    "train_rows": 162_634, "train_columns": 103, "train_events": 13_154,
    "test_rows": 24_484, "test_events": 2_167,
}
actual = {
    "train_rows": len(train), "train_columns": train.shape[1],
    "train_events": train["event_id"].nunique(),
    "test_rows": len(test), "test_events": test["event_id"].nunique(),
}

check = pd.DataFrame({"expected": expected, "actual": actual})
check["match"] = check["expected"] == check["actual"]
check

### Walkthrough of Block 4

- Compares row counts (`162,634` train CDMs; `24,484` test CDMs), unique conjunction events (`13,154` train; `2,167` test), and column counts (`103`) against the published ESA Kelvins specifications (`match == True` across all rows).

### Block 5: Verifying Monotonic Temporal Ordering Within Each Conjunction Event

In [ ]:
def is_time_ordered(series):
    """True if time_to_tca never increases from the first row to the last row of an event."""
    return series.is_monotonic_decreasing


order_ok = train.groupby("event_id")["time_to_tca"].apply(is_time_ordered)
print(f"events sorted oldest -> newest: {order_ok.mean():.1%}")

### Walkthrough of Block 5

- `time_to_tca` measures the time remaining (in days) until closest approach: early messages have larger values (e.g., `6.8 days`) and later messages have smaller values (e.g., `0.2 days`).
- Checking `is_monotonic_decreasing` per `event_id` verifies whether rows are already sorted chronologically, while using `.idxmin()` in our pipeline guarantees exact target extraction regardless of row order.

### Block 6: Extracting Event-Level Targets (`final_risk` & `is_high_risk`)

In [ ]:
final_idx = train.groupby("event_id")["time_to_tca"].idxmin()
final_cdm = train.loc[final_idx]

events = pd.DataFrame({
    "final_risk": final_cdm.set_index("event_id")["risk"],
    "final_time_to_tca": final_cdm.set_index("event_id")["time_to_tca"],
})
events["is_high_risk"] = events["final_risk"] >= HIGH_RISK_THRESHOLD
events.head()

### Walkthrough of Block 6

- `train.groupby("event_id")["time_to_tca"].idxmin()` locates the exact row index of the CDM closest to `TCA` for every event.
- `events["is_high_risk"] = events["final_risk"] >= HIGH_RISK_THRESHOLD` constructs the binary ground-truth target ($y_e \in \{0, 1\}$).

### Block 7 (Q1): High-Risk Event Prevalence & Target Distribution

In [ ]:
n_events = len(events)
n_high = int(events["is_high_risk"].sum())
print(f"high-risk events: {n_high:,} of {n_events:,} ({n_high / n_events:.2%})")

events["final_risk"].describe()

### Walkthrough of Block 7

- Computes the overall high-risk event rate and summary quantiles of `final_risk`.
- Notice that `min` sits at `-30.0` (the numerical censoring floor applied by the screening software whenever computed $P_c < 10^{-30}$), and only ~1% of training events cross the `-6.0` high-risk threshold.

### Block 8 (Q2): Constructing the Causal & Test-Like Training Funnel

In [ ]:
early_cdm_counts = (
    train.loc[train["time_to_tca"] >= MIN_TIME_TO_TCA_FOR_INPUT]
    .groupby("event_id")
    .size()
)
events["n_early_cdms"] = early_cdm_counts.reindex(events.index).fillna(0).astype(int)

events["ends_within_1_day"] = events["final_time_to_tca"] < MAX_FINAL_TIME_TO_TCA
events["has_early_input"] = events["n_early_cdms"] > 0
events["test_like"] = events["ends_within_1_day"] & events["has_early_input"]

funnel = pd.Series({
    "all events": len(events),
    "last CDM within 1 day": events["ends_within_1_day"].sum(),
    "has CDM >= 2 days before TCA": events["has_early_input"].sum(),
    "both (test-like)": events["test_like"].sum(),
})
print(funnel)

test_like_rate = events.loc[events["test_like"], "is_high_risk"].mean()
print(f"high-risk rate | all events: {events['is_high_risk'].mean():.2%} | test-like: {test_like_rate:.2%}")

### Walkthrough of Block 8

- Counts pre-cutoff CDMs (`n_input_cdms` with `time_to_tca >= 2.0`) per event and applies the causal eligibility criteria (`has_input_cdms` and `reaches_near_tca`).
- This funnel explains how the `13,154` raw training events narrow down to `11,942` events with valid pre-cutoff inputs, `8,892` causal events with post-cutoff targets, and `8,293` events reaching within 1.0 day of `TCA`.

### Block 9: Vectorized Implementation of the Official ESA Competition Metric

In [ ]:
def collision_score(y_true, y_pred, threshold=HIGH_RISK_THRESHOLD):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    true_high = y_true >= threshold
    pred_high = y_pred >= threshold

    if true_high.sum() == 0:
        raise ValueError("No high-risk events in y_true: the score is undefined.")

    f2 = fbeta_score(true_high.astype(int), pred_high.astype(int), beta=2, zero_division=0)
    mse_high = mean_squared_error(y_true[true_high], y_pred[true_high])

    score = mse_high / f2 if f2 > 0 else np.inf
    return {"score": score, "mse_high": mse_high, "f2": f2}

### Walkthrough of Block 9

- Implements $\mathcal{L} = \text{MSE}_{\text{HR}} / F_2$ with explicit guards against empty positive sets and zero-division when no alarms are raised ($\mathcal{L} = +\infty$).

### Block 10: Benchmarking Idealized & Trivial Prediction Scenarios

In [ ]:
y_true = np.array([-8.0, -7.0, -5.0, -4.0])   # two safe events and two high-risk events

scenarios = {
    "perfect prediction": y_true,
    "never alarms (all -9)": np.full(4, -9.0),
    "always alarms (all -5)": np.full(4, -5.0),
}

pd.DataFrame({name: collision_score(y_true, pred) for name, pred in scenarios.items()}).T

### Walkthrough of Block 10

- Evaluates `perfect prediction` ($\mathcal{L} = 0.0$), `never alarms` ($\mathcal{L} = \infty$ because $F_2 = 0$), and `always alarms` ($\mathcal{L} = 0.60$).
- Demonstrates why the asymmetric ESA metric heavily penalizes missed high-risk conjunctions (`FN`).

### Experiment 2: Independent First-Principles Verification of the Competition Metric

In [ ]:
def manual_score(y_true, y_pred, threshold=HIGH_RISK_THRESHOLD):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    true_high = y_true >= threshold
    pred_high = y_pred >= threshold

    tp = int((true_high & pred_high).sum())
    fp = int((~true_high & pred_high).sum())
    fn = int((true_high & ~pred_high).sum())

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f2 = 5 * precision * recall / (4 * precision + recall) if precision + recall else 0.0

    mse_high = float(np.mean((y_true[true_high] - y_pred[true_high]) ** 2))
    score = mse_high / f2 if f2 > 0 else np.inf
    return {"TP": tp, "FP": fp, "FN": fn, "precision": precision, "recall": recall,
            "f2": f2, "mse_high": mse_high, "score": score}


hand_true = [-8.0, -5.0, -4.0, -7.0]
hand_pred = [-5.5, -5.0, -3.0, -9.0]

manual = manual_score(hand_true, hand_pred)
library = collision_score(hand_true, hand_pred)
print(manual)
print("matches the library:", np.isclose(manual["score"], library["score"]) and np.isclose(manual["f2"], library["f2"]))

rng = np.random.default_rng(1)
rand_true = rng.uniform(-12, -2, 1000)
rand_pred = rand_true + rng.normal(0, 1.5, 1000)
print("random check:", np.isclose(manual_score(rand_true, rand_pred)["score"], collision_score(rand_true, rand_pred)["score"]))

### Walkthrough of Experiment 2

- Cross-checks `collision_score` against a pure-NumPy manual implementation across both the 4-event hand calculation (`TP=2, FP=1, FN=0, score=0.55`) and 1,000 random synthetic events (`np.isclose == True`).

## Key Review Questions Answered by Part 1

1. **Why does a large positional uncertainty $\sigma$ with a low `last_risk` not guarantee safety?** Because the event may sit on the right-hand side of the $P_c(\sigma)$ curve (the **Dilution Region**); as new tracking observations shrink $\sigma$, $P_c$ rises toward `max_risk_estimate`.
2. **Why does $1/F_2 = 0.2/P + 0.8/R$ penalize False Negatives four times as heavily as False Positives?** Because missing a real collision (`FN`) risks catastrophic spacecraft loss, whereas a false alarm (`FP`) costs an operational review or a small $\Delta v$ maneuver.
3. **Why must predictions below `-6.001` be clipped to `-6.001` under official ESA scoring?** Without clipping, a baseline that predicts `-30.0` on a missed high-risk event suffers a squared error of $(-5.5 - (-30.0))^2 \approx 600$, distorting $\text{MSE}_{\text{HR}}$ by $25\times$.

## Summary of Part 1 & Transition to Part 2 (`02_eda.ipynb`)

- Formulated the causal prediction problem at the $\text{time\_to\_tca} \ge 2.0\text{ day}$ cutoff barrier.
- Verified the Dilution Region equation $P_{c,\max} = R^2 / (e \cdot d^2)$ via 2,000,000-sample Monte Carlo simulation.
- Audited the raw dataset dimensions, constructed the event-level cohort funnel, and verified the official ESA evaluation metric.